# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze data from the FAIR<sup>2</sup> dataset package using the `mlcroissant` library.

### Dataset Source
The dataset structure is defined by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant pandas matplotlib

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load metadata and dataset object
dataset = mlc.Dataset(croissant_url)
# Note: dataset.metadata is an object, not a dict
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Let's review what data is available by listing record sets and previewing the schema structure.

All entities are referenced strictly by their `@id` fields for reliable, standards-based referencing.

In [ ]:
# List all available record sets and their fields using `@id`s
from collections import defaultdict

record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record sets available in the dataset:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        field_ids = [f['@id'] for f in rs['field']] if isinstance(rs['field'], list) else [rs['field']['@id']]
        print(f"   Fields: {field_ids}")

Below, for illustration, we display the first few records from each record set (by @id):

In [ ]:
# Preview sample records from each record set, referencing them by @id
for record_set in record_sets:
    rsid = record_set['@id']
    print(f"Sample records from RecordSet {rsid}:")
    try:
        records = list(dataset.records(record_set=rsid))
        for i, rec in enumerate(records[:3]):
            print(f"  Record {i+1}: {rec}")
        if len(records) == 0:
            print("  (No records available)")
    except Exception as e:
        print(f"  [Error accessing records for {rsid}]: {e}")
    print()

## 3. Data Extraction
Load records from each record set into pandas DataFrames. Use the record set and field `@id` values.

In [ ]:
# For each record set (referenced by @id), extract rows into a DataFrame

# Build a dictionary mapping record set @id to DataFrame with rows
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rsid in record_set_ids:
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rsid)))
        dataframes[rsid] = df
        print(f"Loaded {len(df)} rows from RecordSet {rsid}.")
        print(f"Columns (@id): {list(df.columns)}\n")
    except Exception as e:
        print(f"[Skipped {rsid}] Error loading data: {e}")
        dataframes[rsid] = pd.DataFrame()

# Display the head of the first non-empty DataFrame
for rsid, df in dataframes.items():
    if not df.empty:
        print(f"First five rows from RecordSet {rsid}:")
        display(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: numerical normalizations, filtering, or grouping.

Entities are referenced by their `@id`; for clarity replace `<example_record_set_id>`, `<numeric_field_id>` and `<group_field_id>` as needed.

In [ ]:
# Choose a record set with numeric fields for analysis
# For illustration, pick the first non-empty DataFrame and scan for numeric fields
import numpy as np

eda_rs_id = None
numeric_field = None

for rsid, df in dataframes.items():
    if not df.empty:
        # Find the first numeric field (try casting to float to check)
        for col in df.columns:
            try:
                if pd.api.types.is_numeric_dtype(df[col]) or np.issubdtype(df[col].dropna().astype(float).dtype, np.number):
                    numeric_field = col
                    eda_rs_id = rsid
                    break
            except Exception:
                continue
    if numeric_field is not None:
        break

if eda_rs_id is None or numeric_field is None:
    print("No suitable numeric field found in loaded record sets.")
else:
    df = dataframes[eda_rs_id].copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Select threshold as mean + 1 std, for illustrative filtering
    mean_val = df[numeric_field].mean()
    std_val = df[numeric_field].std()
    threshold = mean_val + std_val
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records in RecordSet {eda_rs_id} where {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    if not filtered_df.empty:
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - mean_val) / std_val
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
    
    # Try to group by another field (pick first non-numeric field)
    group_field = None
    for col in df.columns:
        if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field is not None and not filtered_df.empty:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}: (showing first 5 groups)")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found.")

## 5. Visualization
Below, we create a simple plot (histogram and boxplot) of the selected numeric field in the record set, using `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt
if eda_rs_id is not None and numeric_field is not None and not df[numeric_field].isna().all():
    plt.figure(figsize=(12, 5))
    plt.subplot(1,2,1)
    df[numeric_field].hist(bins=30, color='skyblue', edgecolor='k')
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    
    plt.subplot(1,2,2)
    df.boxplot(column=numeric_field)
    plt.title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load a FAIR<sup>2</sup>-structured dataset via Croissant schema with `mlcroissant`, explored the dataset using record and field `@id` references, extracted records into pandas DataFrames, performed basic cleaning and exploratory analysis, and visualized key numeric columns. This workflow provides a reproducible and standards-based approach for working with structured ML data packages.

**Next steps:** For advanced analyses, refer to the field and record set `@id` values to scale your exploration to new or related Croissant-based datasets!